### Setup: Configure Gemini API
To perform these tasks, we will use the Gemini API. Make sure you have added your `GOOGLE_API_KEY` to the Colab Secrets (the 🔑 icon on the left).

In [28]:
import google.generativeai as genai
from google.colab import userdata

try:
    GOOGLE_API_KEY = userdata.get('GOOGLE_API_KEY')
    genai.configure(api_key=GOOGLE_API_KEY)

    # List all available models to find standard production versions
    available_models = [m.name for m in genai.list_models() if 'generateContent' in m.supported_generation_methods]

    # Filtering for the most stable and modern options
    # We avoid 'preview' and 'robotics' specific models to ensure compatibility
    preferred_models = [
        'models/gemini-1.5-flash',
        'models/gemini-1.5-pro',
        'models/gemini-pro'
    ]

    selected_model = None
    for pref in preferred_models:
        if pref in available_models:
            selected_model = pref
            break

    if not selected_model and available_models:
        selected_model = available_models[0]

    if selected_model:
        model = genai.GenerativeModel(selected_model)
        # Quick test call
        model.generate_content('test')
        print(f"Gemini API configured successfully using model: {selected_model}")
    else:
        print("No suitable models found supporting generateContent.")
except Exception as e:
    print(f"Error configuring Gemini API: {e}")

Gemini API configured successfully using model: models/gemini-2.5-flash


### Step 1: Craft the Control Prompt
We will define the input text and the prompt constraints (tone, format, length, and paraphrasing).

In [29]:
input_text = """Employees must ensure that all remote access to internal systems is established via the approved secure VPN. Under no circumstances should unsecured connections or personal devices lacking endpoint protection be used to access proprietary data or sensitive communications."""

try:
    prompt_v1 = f"""
    Using the input text provided below, rewrite it according to these rules:
    1. Tone: Friendly and clear.
    2. Format: Use bullet points.
    3. Technique: Paraphrase the original (do not use direct quotes).
    4. Constraint: Stay under 75 words total.

    Input Text:
    {input_text}
    """

    # Use the 'model' object initialized in the setup cell
    response = model.generate_content(prompt_v1)
    print("--- Generated Output ---")
    print(response.text)
    print(f"\nWord Count: {len(response.text.split())}")
except Exception as e:
    print(f"Generation Error: {e}")

--- Generated Output ---
Here's the rewritten text:

*   Please use the approved secure VPN for all remote access to internal systems.
*   Avoid using unsecured connections or personal devices without endpoint protection when accessing company data or communications.

Word Count: 35


### Step 3: Mitigate Hallucinations
If the model added details not found in the text (like specific VPN software names), we use a more restrictive prompt.

In [33]:
prompt_v2 = f"""
Rewrite the following text into friendly bullet points under 75 words.
STRICT RULE: Only paraphrase the content provided. Do NOT add new recommendations, external links, or technologies not mentioned in the text.

Input Text:
{input_text}
"""

try:
    # Use the 'model' object initialized in the setup cell
    response_v2 = model.generate_content(prompt_v2)
    print("--- Strict Output ---")
    print(response_v2.text)
except Exception as e:
    print(f"Error: {e}")

--- Strict Output ---
Here are the guidelines for secure remote access:

*   Always use the approved secure VPN for remote access to our internal systems.
*   Avoid using unsecured connections when accessing proprietary data or sensitive communications.
*   Ensure any personal devices used for company access have proper endpoint protection.


### Step 4 & 5: Paraphrasing for Interns and Quote Extraction
Now we target a specific audience and practice extraction.

In [32]:
import time

# Junior Intern Audience
prompt_intern = f"""
Rewrite the input text for a junior intern audience.
- Use plain, supportive language.
- Use no more than 4 bullet points.
- Avoid corporate or legal jargon.

Input Text: {input_text}
"""

# Quote Extraction
prompt_quote = f"""
Extract one direct quote from the following text that best captures the core security policy.

Input Text: {input_text}
"""

def safe_generate(prompt):
    for attempt in range(3):
        try:
            res = model.generate_content(prompt)
            return res.text
        except Exception as e:
            if "429" in str(e):
                print("Rate limit reached. Waiting 35 seconds before retrying...")
                time.sleep(35)
            else:
                return f"Error: {e}"
    return "Failed after retries."

try:
    print("--- Intern Version ---")
    print(safe_generate(prompt_intern))
    print("\n--- Core Quote ---")
    print(safe_generate(prompt_quote))
except Exception as e:
    print(f"Final Error: {e}")

--- Intern Version ---
Here's a simpler way to think about it for your internship:

*   When you need to access company systems or files from home, **always use our secure company VPN**. This special connection helps keep our work safe!
*   Please **never use public Wi-Fi or regular, unsecured internet connections** to do company work.
*   If you're using your own laptop or phone for work, **make sure it has our approved company security software** installed.
*   Following these steps helps us **protect all our important company information and private communications.**

--- Core Quote ---
"Under no circumstances should unsecured connections or personal devices lacking endpoint protection be used to access proprietary data or sensitive communications."
